# NLP Assignment 2: Word Segmentation

**Name:** Ritweek Raj  
**Roll Number:** U24AI067  
**Assignment:** Word Segmentation using Greedy and Dynamic Programming Algorithms  

---

This notebook implements:
1. Greedy (longest-match) word segmentation.
2. Dynamic Programming segmentation using log probabilities derived from word frequencies.
3. Levenshtein edit distance and accuracy metrics.
4. Evaluation of both approaches on a dataset of 1000 test cases.

## Step 1: Import Libraries
First, we import the necessary standard python libraries.

In [1]:
import json
import math
import time

## Step 2: Load JSON Dataset
We load the JSON dataset file `text_segmentation_dataset.json` containing the metadata, vocabulary frequencies (`word_counts`), and the test cases.

In [2]:
# Load the dataset
dataset_path = "text_segmentation_dataset.json"
with open(dataset_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Extract key elements
metadata = data["metadata"]
word_freq = data["word_counts"]
test_cases = data["test_cases"]
total_words = metadata["total_corpus_words"]

print("Dataset loaded successfully!")

Dataset loaded successfully!


## Step 3: Inspect Metadata
Let's display the metadata information including vocabulary size, total corpus words, and test case count.

In [3]:
print("--- Dataset Metadata ---")
print(f"Vocabulary Size: {metadata['vocabulary_size']}")
print(f"Total Corpus Words: {metadata['total_corpus_words']}")
print(f"Test Case Count: {metadata['test_case_count']}")

--- Dataset Metadata ---
Vocabulary Size: 1500
Total Corpus Words: 735040
Test Case Count: 1000


## Step 4: Create Vocabulary
We store vocabulary frequencies in a HashMap/Dictionary for $O(1)$ lookups and create a set of vocabulary words.

In [4]:
# Create dictionary for O(1) lookup frequencies
word_freq_dict = dict(word_freq)

# Create vocabulary set
vocab = set(word_freq_dict.keys())

print(f"Vocabulary set created with {len(vocab)} unique words.")

Vocabulary set created with 1500 unique words.


## Step 5: Compute Word Log Probabilities
Convert word frequencies into log-probabilities. Under a unigram language model:
$$P(\text{word}) = \frac{\text{frequency(word)}}{\text{Total Corpus Words}}$$
Using base-10 log, we define:
$$\log_{10} P(\text{word}) = \log_{10}(\text{frequency(word)}) - \log_{10}(\text{Total Corpus Words})$$
This avoids floating point underflow when multiplying probabilities.

For Out-of-Vocabulary (OOV) words, we define a penalty function proportional to length, ensuring vocabulary words are strongly preferred but arbitrary segmentations can still be fallback-evaluated if needed.

In [5]:
# Compute log probabilities
log_probability = {}
for word, freq in word_freq_dict.items():
    log_probability[word] = math.log10(freq / total_words)

# Out-of-vocabulary (OOV) probability function
def get_word_log_prob(word):
    if word in log_probability:
        return log_probability[word]
    else: 
        # A length-based penalty for OOV words.
        # -15.0 log penalty per character acts as a very low probability.
        return -15.0 * len(word)

## Step 6: Implement Greedy Segmentation
The Greedy algorithm attempts to segment the string by matching the longest possible prefix from the vocabulary at each position.
If a match is found, it appends the word and advances the index. If no match is found, it consumes a single character and advances by 1.

In [6]:
def greedy_segmentation(text, vocab):
    words = []
    i = 0
    n = len(text)
    
    # Calculate the maximum word length in our vocabulary to optimize the prefix search
    max_len = max(len(w) for w in vocab) if vocab else 0
    
    while i < n:
        match_found = False
        # Try prefixes from longest possible to shortest
        for l in range(min(n - i, max_len), 0, -1):
            substring = text[i:i+l]
            if substring in vocab:
                words.append(substring)
                i += l
                match_found = True
                break
        if not match_found:
            # Fallback to taking a single character
            words.append(text[i])
            i += 1
            
    return " ".join(words)

## Step 7: Test Greedy on One Example
Let's test our Greedy segmentation implementation on the first test case.

In [7]:
sample_input = test_cases[0]["input"]
sample_ground_truth = test_cases[0]["ground_truth"]
greedy_output = greedy_segmentation(sample_input, vocab)

print(f"Input:        {sample_input}")
print(f"Ground Truth: {sample_ground_truth}")
print(f"Greedy Pred:  {greedy_output}")

Input:        itthatthecitytakestepstothisproblem
Ground Truth: it that the city take steps to this problem
Greedy Pred:  it that the city takes t e p s to this problem


## Step 8: Implement Dynamic Programming Segmentation
Instead of greedily choosing the longest word, the Dynamic Programming approach finds the segmentation that maximizes the sum of log-probabilities of the segmented words.

Let `dp[i]` be the maximum log-probability of the segmentation of the suffix `text[i:]`.
Base Case:
`dp[n] = 0.0`
Recurrence:
`dp[i] = max_{j > i} (log P(text[i:j]) + dp[j])`

We store the optimal split boundaries in `parent` to reconstruct the optimal sequence of words.

In [8]:
def dp_segmentation(text, word_counts, total_corpus_words):
    n = len(text)
    
    # dp[i] will store the maximum log-probability of the best segmentation from index i to end.
    dp = [float('-inf')] * (n + 1)
    # parent[i] will store the ending index j of the first word in the optimal segmentation of suffix text[i:]
    parent = [-1] * (n + 1)
    
    # Base case: empty suffix
    dp[n] = 0.0
    
    max_word_len = max(len(w) for w in word_counts.keys()) if word_counts else 0
    
    # Run DP backward
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, min(n + 1, i + max_word_len + 1)):
            word = text[i:j]
            word_prob = get_word_log_prob(word)
            score = word_prob + dp[j]
            
            if score > dp[i]:
                dp[i] = score
                parent[i] = j
                
        # Fallback for OOV characters
        if dp[i] == float('-inf'):
            word = text[i:i+1]
            dp[i] = get_word_log_prob(word) + dp[i+1]
            parent[i] = i + 1
            
    # Reconstruct optimal segmentation
    words = []
    i = 0
    while i < n:
        j = parent[i]
        if j == -1 or j <= i:
            words.append(text[i])
            i += 1
        else:
            words.append(text[i:j])
            i = j
            
    return " ".join(words)

## Step 9: Test DP on One Example
Let's test our Dynamic Programming segmentation implementation on the first test case.

In [9]:
dp_output = dp_segmentation(sample_input, word_freq_dict, total_words)

print(f"Input:        {sample_input}")
print(f"Ground Truth: {sample_ground_truth}")
print(f"DP Pred:      {dp_output}")

Input:        itthatthecitytakestepstothisproblem
Ground Truth: it that the city take steps to this problem
DP Pred:      it that the city take steps to this problem


## Step 10: Implement Edit Distance
We implement the standard Dynamic Programming algorithm for Levenshtein distance (character-level, including spaces).

In [10]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        s1, s2 = s2, s1
        m, n = n, m
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                curr[j] = prev[j-1]
            else:
                curr[j] = min(prev[j] + 1, curr[j-1] + 1, prev[j-1] + 1)
        prev = curr
    return prev[n]

## Step 11: Implement Accuracy Calculation
We implement a helper function to calculate the percentage of exact matches between the predicted and ground truth segmentations.

In [11]:
def calculate_accuracy(predictions, ground_truths):
    correct = sum(1 for p, g in zip(predictions, ground_truths) if p == g)
    return (correct / len(ground_truths)) * 100

## Step 12: Evaluate Greedy on All 1000 Test Cases
We run the Greedy algorithm on all 1000 test cases and calculate the Accuracy and Average Edit Distance.

In [12]:
greedy_predictions = []
greedy_distances = []
ground_truths = [tc["ground_truth"] for tc in test_cases]

start_time = time.time()
for tc in test_cases:
    pred = greedy_segmentation(tc["input"], vocab)
    greedy_predictions.append(pred)
    greedy_distances.append(edit_distance(pred, tc["ground_truth"]))
greedy_time = time.time() - start_time

greedy_accuracy = calculate_accuracy(greedy_predictions, ground_truths)
greedy_avg_distance = sum(greedy_distances) / len(test_cases)

print(f"Greedy Evaluation Completed in {greedy_time:.4f} seconds.")
print(f"Greedy Accuracy: {greedy_accuracy:.2f}%")
print(f"Greedy Average Edit Distance: {greedy_avg_distance:.4f}")

Greedy Evaluation Completed in 0.1902 seconds.
Greedy Accuracy: 69.10%
Greedy Average Edit Distance: 1.2900


## Step 13: Evaluate DP on All 1000 Test Cases
We run the Dynamic Programming algorithm on all 1000 test cases and calculate the Accuracy and Average Edit Distance.

In [13]:
dp_predictions = []
dp_distances = []

start_time = time.time()
for tc in test_cases:
    pred = dp_segmentation(tc["input"], word_freq_dict, total_words)
    dp_predictions.append(pred)
    dp_distances.append(edit_distance(pred, tc["ground_truth"]))
dp_time = time.time() - start_time

dp_accuracy = calculate_accuracy(dp_predictions, ground_truths)
dp_avg_distance = sum(dp_distances) / len(test_cases)

print(f"DP Evaluation Completed in {dp_time:.4f} seconds.")
print(f"DP Accuracy: {dp_accuracy:.2f}%")
print(f"DP Average Edit Distance: {dp_avg_distance:.4f}")

DP Evaluation Completed in 0.2463 seconds.
DP Accuracy: 98.20%
DP Average Edit Distance: 0.0280


## Step 14: Display Sample Predictions
Let's print the 1000 test case predictions in a clear table format to compare the predictions side-by-side.

In [14]:
print(f"{'Input':<45} | {'Ground Truth':<45} | {'Greedy Prediction':<45} | {'DP Prediction':<45}")
print("-" * 190)
for i in range(1000):
    inp = test_cases[i]["input"]
    gt = test_cases[i]["ground_truth"]
    g_pred = greedy_predictions[i]
    d_pred = dp_predictions[i]
    print(f"{inp:<45} | {gt:<45} | {g_pred:<45} | {d_pred:<45}")

Input                                         | Ground Truth                                  | Greedy Prediction                             | DP Prediction                                
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
itthatthecitytakestepstothisproblem           | it that the city take steps to this problem   | it that the city takes t e p s to this problem | it that the city take steps to this problem  
oftitlelawwasalsobythe                        | of title law was also by the                  | of title law was also by the                  | of title law was also by the                 
failuretodothiswillcontinuetoplaceaon         | failure to do this will continue to place a on | failure to do this will continue top l a c e a on | failure to do this will continue to place a on
onotherthethat                            

## Step 15: Display Final Comparison Table
We summarize the overall statistics in a markdown/ASCII table comparing Greedy and DP algorithms.

In [15]:
print("=" * 60)
print(f"{'Metric':<30} | {'Greedy Algorithm':<16} | {'DP Algorithm':<16}")
print("-" * 60)
print(f"{'Accuracy (%)':<30} | {greedy_accuracy:<16.2f} | {dp_accuracy:<16.2f}")
print(f"{'Average Edit Distance':<30} | {greedy_avg_distance:<16.4f} | {dp_avg_distance:<16.4f}")
print(f"{'Execution Time (seconds)':<30} | {greedy_time:<16.4f} | {dp_time:<16.4f}")
print("=" * 60)

Metric                         | Greedy Algorithm | DP Algorithm    
------------------------------------------------------------
Accuracy (%)                   | 69.10            | 98.20           
Average Edit Distance          | 1.2900           | 0.0280          
Execution Time (seconds)       | 0.1902           | 0.2463          


## Step 16: Write Conclusions
### Comparison & Findings:
1. **Accuracy**: The **Dynamic Programming** approach achieves an accuracy of **98.20%**, significantly outperforming the **Greedy** approach which only achieves **69.10%**.
2. **Edit Distance**: The DP approach achieves an average edit distance of **0.028**, while the Greedy approach has an average edit distance of **1.2900**. A lower edit distance indicates that the DP segmentations are much closer to the ground truth.
3. **Why DP outperforms Greedy**:
   - The Greedy approach only makes local decisions by matching the longest available prefix at the current start index. This often results in matching a long word that leaves a sequence of characters that cannot be segmented correctly, or choosing a long word when a combination of shorter, more frequent words would be correct.
   - For example, with an input containing a substring that can be segmented into two common words, the Greedy algorithm might pick a single rare, long word if it matches a prefix, or it might segment in a way that leaves incorrect remaining characters.
   - The Dynamic Programming approach optimizes the segmentation globally. It computes the segmentation that maximizes the product of unigram word probabilities (sum of log probabilities) derived from the corpus counts. This allows the DP algorithm to weigh a single long word against a sequence of shorter, highly probable words and select the globally optimal segmentation.